## Install (Colab Only)

In [ ]:
# install
!pip install pyepo
!pip install gurobipy

Remove the problematic hook from Google Colab.

In [ ]:
import sys
sys.meta_path = [hook for hook in sys.meta_path if not any(keyword in str(hook) for keyword in ["google.colab"])]

# 04 - CaVE for Binary Linear Programs

This notebook introduces **CaVE** (Cone-Aligned Vector Estimation), a fast
end-to-end loss for predict-then-optimize problems where the inner
optimization is a **binary linear program (BLP)** - shortest path, knapsack,
TSP, vehicle routing, etc.

> Reference: Tang & Khalil, *CaVE: A Cone-Aligned Approach for Fast
> Predict-then-optimize with Binary Linear Programs*, CPAIOR 2024.
> Paper: <https://arxiv.org/abs/2312.07718> Repo: <https://github.com/khalil-research/CaVE>


## 1 The idea in one paragraph

Most decision-focused losses (SPO+, DPO, PFYL, ...) call the solver **once
per training step** to turn the predicted cost into a decision. That
per-step solve is the bottleneck at scale.

**CaVE skips it.** For each training instance, the true optimal vertex is
the intersection of a small set of *binding* (tight) constraints. The
cost vectors that pick that vertex form a polyhedral cone - the normal
cone at the optimum. CaVE pre-computes the binding-constraint normals
once during data prep, then at every training step it just projects the
predicted cost onto that cone and minimizes
$1 - \cos(-\hat{\mathbf{c}}, \mathrm{proj})$.

No solver call per step. The projection is a small, batched APGD that runs
natively on GPU.

**CaVE vs CaVE+.** The paper proposes two presets:
- **CaVE** runs APGD to a tolerance (slower, lands on the cone boundary).
- **CaVE+** truncates APGD at 20 iterations (faster, lands strictly inside
  the cone, no quality loss in practice). This is what `coneAlignedCosine`
  uses by default.

**When to use it.**
- The inner problem must be a **binary LP** (sols in {0,1}^n). Continuous
  or general integer problems are out of scope.
- The headline speedup is realized on **GPU**. The projection is a
  `torch.compile`'d APGD designed for CUDA - on CPU the compile overhead
  dominates and CaVE can end up no faster than (or slower than) SPO+.


## 2 Setup

Standard imports plus a device check. We seed everything so the runs are
reproducible.


In [ ]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import pyepo
from pyepo.data.dataset import optDataset, optDatasetConstrs, collate_tight_constraints
from pyepo.model.grb import tspDFJModel

torch.manual_seed(0)
np.random.seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type != "cuda":
    print("[warning] CaVE+ is designed for GPU; on CPU it will not be faster than SPO+.")


# linear predictor: features -> per-edge cost
class LinearPred(nn.Module):
    def __init__(self, num_feat, num_cost):
        super().__init__()
        self.linear = nn.Linear(num_feat, num_cost)

    def forward(self, x):
        return self.linear(x)


## 3 Training Helpers

Two helpers: one for **SPO+** (needs `(x, c, w, z)`) and one for **CaVE+** (needs `(x, c, w, z, tight_ctrs)`). Both record the per-epoch test regret and a cumulative wall-clock timestamp so we can plot regret-vs-time alongside regret-vs-epoch. Each helper re-seeds the predictor init for a fair head-to-head.

In [ ]:
def regret(predmodel, optmodel, loader):
    return pyepo.metric.regret(predmodel, optmodel, loader)


def train_spo(optmodel, train_loader, test_loader, num_feat, num_epochs=10, lr=1e-2):
    torch.manual_seed(0)
    pred = LinearPred(num_feat, optmodel.num_cost).to(device)
    opt = torch.optim.Adam(pred.parameters(), lr=lr)
    loss_fn = pyepo.func.SPOPlus(optmodel, processes=1)
    t0 = time.time()
    regrets = [regret(pred, optmodel, test_loader)]
    times = [0.0]
    for ep in range(num_epochs):
        for x, c, w, z in train_loader:
            x, c, w, z = x.to(device), c.to(device), w.to(device), z.to(device)
            cp = pred(x)
            loss = loss_fn(cp, c, w, z)
            opt.zero_grad()
            loss.backward()
            opt.step()
        regrets.append(regret(pred, optmodel, test_loader))
        times.append(time.time() - t0)
    return regrets, times


def train_cave(optmodel, train_loader, test_loader, num_feat, num_epochs=10, lr=1e-2):
    torch.manual_seed(0)
    pred = LinearPred(num_feat, optmodel.num_cost).to(device)
    opt = torch.optim.Adam(pred.parameters(), lr=lr)
    # defaults reproduce the CaVE+ truncated-APGD preset
    loss_fn = pyepo.func.coneAlignedCosine(optmodel, processes=1)
    t0 = time.time()
    regrets = [regret(pred, optmodel, test_loader)]
    times = [0.0]
    for ep in range(num_epochs):
        for x, c, w, z, tight_ctrs in train_loader:
            x, c, w, z = x.to(device), c.to(device), w.to(device), z.to(device)
            tight_ctrs = tight_ctrs.to(device)
            cp = pred(x)
            loss = loss_fn(cp, tight_ctrs)
            opt.zero_grad()
            loss.backward()
            opt.step()
        regrets.append(regret(pred, optmodel, test_loader))
        times.append(time.time() - t0)
    return regrets, times

## 4 Experiment A: 20-node TSP

Matches the canonical CaVE sample code (TSP-20 with 100 training instances, 10 features, polynomial degree 4, noise width 0.5).

In [ ]:
NUM_DATA, NUM_TEST, NUM_FEAT, NUM_NODES = 100, 50, 10, 20
BATCH, EPOCHS = 32, 10

# generate data (CaVE paper / README sample settings: deg=4, noise=0.5)
x_train, c_train = pyepo.data.tsp.genData(NUM_DATA, NUM_FEAT, NUM_NODES, deg=4, noise_width=0.5, seed=42)
x_test, c_test = pyepo.data.tsp.genData(NUM_TEST, NUM_FEAT, NUM_NODES, deg=4, noise_width=0.5, seed=43)

# DFJ formulation (lazy subtour elimination)
optmodel = tspDFJModel(num_nodes=NUM_NODES)

# SPO+ uses optDataset (sols + objs)
ds_train = optDataset(optmodel, x_train, c_train)
ds_test = optDataset(optmodel, x_test, c_test)
loader_spo = DataLoader(ds_train, batch_size=BATCH, shuffle=True)
loader_test = DataLoader(ds_test, batch_size=BATCH, shuffle=False)

# CaVE uses optDatasetConstrs (sols + objs + tight constraint normals at the optimum)
ds_train_c = optDatasetConstrs(optmodel, x_train, c_train)
loader_cave = DataLoader(ds_train_c, batch_size=BATCH, shuffle=True,
                         collate_fn=collate_tight_constraints)

In [ ]:
spo_regrets_20, spo_times_20 = train_spo(optmodel, loader_spo, loader_test, NUM_FEAT, EPOCHS)
cave_regrets_20, cave_times_20 = train_cave(optmodel, loader_cave, loader_test, NUM_FEAT, EPOCHS)

print(f'SPO+  : final regret={spo_regrets_20[-1]:.4f}, wall={spo_times_20[-1]:.1f}s')
print(f'CaVE+ : final regret={cave_regrets_20[-1]:.4f}, wall={cave_times_20[-1]:.1f}s')
print(f'Speedup (SPO+ wall / CaVE+ wall): {spo_times_20[-1] / cave_times_20[-1]:.2f}x')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
epochs = list(range(EPOCHS + 1))
ax1.plot(epochs, spo_regrets_20, marker='o', label='SPO+')
ax1.plot(epochs, cave_regrets_20, marker='s', label='CaVE+')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test regret')
ax1.set_title(f'TSP-{NUM_NODES}: regret vs epoch')
ax1.grid(alpha=0.3); ax1.legend()

ax2.plot(spo_times_20, spo_regrets_20, marker='o', label='SPO+')
ax2.plot(cave_times_20, cave_regrets_20, marker='s', label='CaVE+')
ax2.set_xlabel('Wall-clock time (s)'); ax2.set_ylabel('Test regret')
ax2.set_title(f'TSP-{NUM_NODES}: regret vs time')
ax2.grid(alpha=0.3); ax2.legend()
plt.tight_layout(); plt.show()

## 5 Experiment B: 50-node TSP

Same recipe with twice as many cities. SPO+ pays a heavier per-step cost because every gradient step solves the surrogate TSP; CaVE+ only pays the cone projection (a few APGD iterations on GPU/CPU). The wall-clock plot makes this gap visible.

In [ ]:
NUM_DATA_B, NUM_TEST_B, NUM_NODES_B = 50, 20, 50
EPOCHS_B = 10

x_train_b, c_train_b = pyepo.data.tsp.genData(NUM_DATA_B, NUM_FEAT, NUM_NODES_B, deg=4, noise_width=0.5, seed=44)
x_test_b, c_test_b = pyepo.data.tsp.genData(NUM_TEST_B, NUM_FEAT, NUM_NODES_B, deg=4, noise_width=0.5, seed=45)

optmodel_b = tspDFJModel(num_nodes=NUM_NODES_B)

ds_train_b = optDataset(optmodel_b, x_train_b, c_train_b)
ds_test_b = optDataset(optmodel_b, x_test_b, c_test_b)
loader_spo_b = DataLoader(ds_train_b, batch_size=BATCH, shuffle=True)
loader_test_b = DataLoader(ds_test_b, batch_size=BATCH, shuffle=False)

ds_train_cb = optDatasetConstrs(optmodel_b, x_train_b, c_train_b)
loader_cave_b = DataLoader(ds_train_cb, batch_size=BATCH, shuffle=True,
                           collate_fn=collate_tight_constraints)

In [ ]:
spo_regrets_50, spo_times_50 = train_spo(optmodel_b, loader_spo_b, loader_test_b, NUM_FEAT, EPOCHS_B)
cave_regrets_50, cave_times_50 = train_cave(optmodel_b, loader_cave_b, loader_test_b, NUM_FEAT, EPOCHS_B)

print(f'SPO+  : final regret={spo_regrets_50[-1]:.4f}, wall={spo_times_50[-1]:.1f}s')
print(f'CaVE+ : final regret={cave_regrets_50[-1]:.4f}, wall={cave_times_50[-1]:.1f}s')
print(f'Speedup (SPO+ wall / CaVE+ wall): {spo_times_50[-1] / cave_times_50[-1]:.2f}x')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
epochs_b = list(range(EPOCHS_B + 1))
ax1.plot(epochs_b, spo_regrets_50, marker='o', label='SPO+')
ax1.plot(epochs_b, cave_regrets_50, marker='s', label='CaVE+')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test regret')
ax1.set_title(f'TSP-{NUM_NODES_B}: regret vs epoch')
ax1.grid(alpha=0.3); ax1.legend()

ax2.plot(spo_times_50, spo_regrets_50, marker='o', label='SPO+')
ax2.plot(cave_times_50, cave_regrets_50, marker='s', label='CaVE+')
ax2.set_xlabel('Wall-clock time (s)'); ax2.set_ylabel('Test regret')
ax2.set_title(f'TSP-{NUM_NODES_B}: regret vs time')
ax2.grid(alpha=0.3); ax2.legend()
plt.tight_layout(); plt.show()

## 6 Takeaways

* **CaVE+** matches (or closely tracks) SPO+ on test regret, but is significantly faster per gradient step because the backward path is a small batched cone projection instead of a TSP solve.
* **The supervision changes**: SPO+ wants `(c_true, w_true, z_true)`. CaVE wants the binding-constraint normals at the optimum, prepared via `optDatasetConstrs` + `collate_tight_constraints`.
* **Scope**: CaVE is defined for *binary linear programs*. `optDatasetConstrs` raises if a training-instance optimum is not binary.

**Tuning `pyepo.func.coneAlignedCosine`**:

* Defaults `max_iters=20, tol_grad=None` = **CaVE+** (truncated APGD, lands inside the cone). Recommended.
* Pass `max_iters=None, tol_grad=1e-4` to approach the original **CaVE** preset (full APGD convergence on the cone boundary). Slower and usually no regret gain.